# 🏥 ACL Tear Detection - Test New MRI Data

This notebook provides a clean, user-friendly interface to test new MRI scans using your trained ACL tear detection model.

---

## 📋 Quick Guide

1. **Single Prediction**: Test one MRI scan at a time with detailed visualization
2. **Batch Prediction**: Test multiple scans at once and get a summary report
3. **Interactive Mode**: Enter file paths and get instant predictions

---

## Step 1: Setup & Load Model

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from skimage.transform import resize
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Configuration
MODEL_PATH = "acl_detector_model.pth"
DATA_DIR = "DATASET/MRI"
METADATA_PATH = "DATASET/MRI/metadata.csv"
TARGET_SIZE = (16, 128, 128)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Using device: {device}")
print(f"📁 Model path: {MODEL_PATH}")
print(f"📁 Data directory: {DATA_DIR}")

🖥️  Using device: cpu
📁 Model path: acl_detector_model.pth
📁 Data directory: DATASET/MRI


In [3]:
# Define the model architecture (must match trained model)
class ACLDetector3D(nn.Module):
    def __init__(self):
        super(ACLDetector3D, self).__init__()
        
        self.conv1 = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2)
        )
        
        self.conv4 = nn.Sequential(
            nn.Conv3d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm3d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.classifier(x)
        return x

# Load the trained model
model = ACLDetector3D().to(device)

# Load checkpoint (model was saved with optimizer state and history)
checkpoint = torch.load(MODEL_PATH, map_location=device)

# Extract model weights from checkpoint
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"📊 Loaded from checkpoint (Best val loss: {checkpoint.get('best_val_loss', 'N/A'):.4f})")
else:
    # If it's just the state dict directly
    model.load_state_dict(checkpoint)

model.eval()
print("✅ Model loaded successfully!")

📊 Loaded from checkpoint (Best val loss: 0.5977)
✅ Model loaded successfully!


## Step 2: Helper Functions

In [4]:
def find_pck_file(data_dir, filename):
    """Search for .pck file in vol01-vol08 folders."""
    for vol_num in range(1, 9):
        vol_folder = f"vol{vol_num:02d}"
        path = os.path.join(data_dir, vol_folder, filename)
        if os.path.exists(path):
            return path
    return None


def load_mri(filepath):
    """Load MRI data from .pck file."""
    with open(filepath, 'rb') as f:
        return pickle.load(f)


def extract_roi(mri, roi_info):
    """Extract Region of Interest from MRI volume."""
    z = int(roi_info['roiZ'])
    y = int(roi_info['roiY'])
    x = int(roi_info['roiX'])
    d = int(roi_info['roiDepth'])
    h = int(roi_info['roiHeight'])
    w = int(roi_info['roiWidth'])
    return mri[z:z+d, y:y+h, x:x+w]


def preprocess_roi(roi, target_size=TARGET_SIZE):
    """Normalize and resize ROI."""
    roi = roi.astype(np.float32)
    roi = (roi - roi.min()) / (roi.max() - roi.min() + 1e-8)
    roi = resize(roi, target_size, mode='constant', anti_aliasing=True, preserve_range=True)
    return roi.astype(np.float32)


def predict_single(model, mri_tensor, device):
    """Make prediction on a single sample."""
    model.eval()
    with torch.no_grad():
        output = model(mri_tensor.to(device))
        probability = output.item()
        prediction = 1 if probability >= 0.5 else 0
    return prediction, probability


print("✅ Helper functions loaded!")

✅ Helper functions loaded!


---

## 🔬 Option 1: Test a Single MRI Scan

Get a detailed prediction with visualization for one scan.

In [ ]:
def test_single_scan(filename, metadata_df, data_dir=DATA_DIR, show_slices=True):
    """
    Test a single MRI scan and display results with visualization.
    
    Parameters:
    -----------
    filename : str
        The .pck filename (e.g., '329637-8.pck')
    metadata_df : DataFrame
        The metadata DataFrame containing ROI information
    data_dir : str
        Path to the MRI data directory
    show_slices : bool
        Whether to display MRI slices
    """
    
    print("\n" + "="*60)
    print(f"🔬 ANALYZING: {filename}")
    print("="*60)
    
    # Find the file
    filepath = find_pck_file(data_dir, filename)
    if filepath is None:
        print(f"❌ Error: Could not find file '{filename}'")
        return None
    
    # Get metadata for this file
    row = metadata_df[metadata_df['volumeFilename'] == filename]
    if len(row) == 0:
        print(f"❌ Error: No metadata found for '{filename}'")
        return None
    row = row.iloc[0]
    
    # Load and process MRI
    print("\n📂 Loading MRI data...")
    mri = load_mri(filepath)
    print(f"   Original shape: {mri.shape}")
    
    roi = extract_roi(mri, row)
    print(f"   ROI shape: {roi.shape}")
    
    roi_processed = preprocess_roi(roi)
    print(f"   Processed shape: {roi_processed.shape}")
    
    # Prepare tensor
    tensor = torch.tensor(roi_processed).unsqueeze(0).unsqueeze(0)  # (1, 1, D, H, W)
    
    # Make prediction
    print("\n🤖 Running model prediction...")
    prediction, probability = predict_single(model, tensor, device)
    
    # Display results
    print("\n" + "-"*60)
    print("📊 PREDICTION RESULTS")
    print("-"*60)
    
    if prediction == 1:
        status = "🔴 ACL TEAR DETECTED"
        color = 'red'
    else:
        status = "🟢 NO ACL TEAR DETECTED"
        color = 'green'
    
    print(f"\n   {status}")
    print(f"   Confidence: {probability*100:.1f}%")
    
    # Show ground truth if available
    if 'aclDiagnosis' in row:
        actual = row['aclDiagnosis']
        actual_label = "ACL Tear" if actual > 0 else "No ACL Tear"
        print(f"\n   Ground Truth: {actual_label} (diagnosis code: {actual})")
        
        is_correct = (prediction == 1 and actual > 0) or (prediction == 0 and actual == 0)
        print(f"   Prediction: {'✅ Correct' if is_correct else '❌ Incorrect'}")
    
    # Visualization
    if show_slices:
        print("\n📷 MRI Visualization:")
        fig, axes = plt.subplots(2, 4, figsize=(14, 7))
        fig.suptitle(f"MRI Scan: {filename}\nPrediction: {status}", fontsize=14, fontweight='bold')
        
        n_slices = roi_processed.shape[0]
        slice_indices = np.linspace(0, n_slices-1, 8, dtype=int)
        
        for idx, ax in enumerate(axes.flat):
            slice_idx = slice_indices[idx]
            ax.imshow(roi_processed[slice_idx], cmap='gray')
            ax.set_title(f"Slice {slice_idx+1}/{n_slices}", fontsize=10)
            ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Confidence gauge
        fig, ax = plt.subplots(figsize=(8, 2))
        ax.barh([0], [probability], color=color, height=0.5, alpha=0.7)
        ax.barh([0], [1-probability], left=[probability], color='lightgray', height=0.5, alpha=0.5)
        ax.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold (50%)')
        ax.set_xlim(0, 1)
        ax.set_yticks([])
        ax.set_xlabel('Prediction Probability', fontsize=12)
        ax.set_title('Confidence Meter', fontsize=12, fontweight='bold')
        ax.legend(loc='upper right')
        
        # Add percentage labels
        ax.text(probability/2, 0, f"{probability*100:.1f}%", ha='center', va='center', 
                fontsize=12, fontweight='bold', color='white')
        
        plt.tight_layout()
        plt.show()
    
    return {
        'filename': filename,
        'prediction': prediction,
        'probability': probability,
        'label': 'ACL Tear' if prediction == 1 else 'No ACL Tear'
    }

print("✅ Single scan test function ready!")

In [ ]:
# Load metadata
metadata = pd.read_csv(METADATA_PATH)
print(f"📋 Loaded metadata with {len(metadata)} records")
print(f"\n📁 Available files (first 10):")
print(metadata['volumeFilename'].head(10).tolist())

In [ ]:
# =====================================================
# 🔧 EDIT THIS: Enter the filename you want to test
# =====================================================

TEST_FILENAME = "329637-8.pck"  # <-- Change this to your file

# Run the test
result = test_single_scan(TEST_FILENAME, metadata)

---

## 📊 Option 2: Test Multiple Scans (Batch Prediction)

Test multiple MRI scans at once and get a summary report.

In [ ]:
def test_batch(filenames, metadata_df, data_dir=DATA_DIR):
    """
    Test multiple MRI scans and generate a summary report.
    
    Parameters:
    -----------
    filenames : list
        List of .pck filenames to test
    metadata_df : DataFrame
        The metadata DataFrame
    data_dir : str
        Path to the MRI data directory
    """
    
    print("\n" + "="*70)
    print(f"📊 BATCH PREDICTION - Testing {len(filenames)} scans")
    print("="*70)
    
    results = []
    
    for i, filename in enumerate(filenames, 1):
        print(f"\n[{i}/{len(filenames)}] Processing: {filename}...", end=" ")
        
        try:
            # Find file
            filepath = find_pck_file(data_dir, filename)
            if filepath is None:
                print("❌ File not found")
                continue
            
            # Get metadata
            row = metadata_df[metadata_df['volumeFilename'] == filename]
            if len(row) == 0:
                print("❌ No metadata")
                continue
            row = row.iloc[0]
            
            # Process
            mri = load_mri(filepath)
            roi = extract_roi(mri, row)
            roi_processed = preprocess_roi(roi)
            tensor = torch.tensor(roi_processed).unsqueeze(0).unsqueeze(0)
            
            # Predict
            prediction, probability = predict_single(model, tensor, device)
            
            # Store result
            result = {
                'Filename': filename,
                'Prediction': 'ACL Tear' if prediction == 1 else 'No Tear',
                'Confidence': f"{probability*100:.1f}%",
                'Probability': probability
            }
            
            # Add ground truth if available
            if 'aclDiagnosis' in row:
                result['Actual'] = 'ACL Tear' if row['aclDiagnosis'] > 0 else 'No Tear'
                result['Correct'] = '✅' if (prediction == 1) == (row['aclDiagnosis'] > 0) else '❌'
            
            results.append(result)
            print(f"{'🔴' if prediction == 1 else '🟢'} {result['Prediction']} ({result['Confidence']})")
            
        except Exception as e:
            print(f"❌ Error: {str(e)}")
    
    # Create results DataFrame
    results_df = pd.DataFrame(results)
    
    # Summary statistics
    print("\n" + "="*70)
    print("📈 SUMMARY REPORT")
    print("="*70)
    
    n_total = len(results_df)
    n_tears = (results_df['Prediction'] == 'ACL Tear').sum()
    n_healthy = n_total - n_tears
    
    print(f"\n   Total scans tested: {n_total}")
    print(f"   🔴 ACL Tears detected: {n_tears} ({n_tears/n_total*100:.1f}%)")
    print(f"   🟢 No tears detected: {n_healthy} ({n_healthy/n_total*100:.1f}%)")
    
    if 'Correct' in results_df.columns:
        n_correct = (results_df['Correct'] == '✅').sum()
        accuracy = n_correct / n_total * 100
        print(f"\n   📊 Accuracy: {accuracy:.1f}% ({n_correct}/{n_total} correct)")
    
    # Display results table
    print("\n" + "-"*70)
    print("📋 DETAILED RESULTS")
    print("-"*70)
    display(results_df.drop(columns=['Probability'], errors='ignore'))
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Pie chart
    colors = ['#ff6b6b', '#51cf66']
    axes[0].pie([n_tears, n_healthy], labels=['ACL Tear', 'No Tear'], 
                autopct='%1.1f%%', colors=colors, startangle=90,
                explode=(0.05, 0))
    axes[0].set_title('Prediction Distribution', fontsize=12, fontweight='bold')
    
    # Confidence distribution
    axes[1].hist(results_df['Probability'], bins=10, color='steelblue', edgecolor='black', alpha=0.7)
    axes[1].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold')
    axes[1].set_xlabel('Prediction Probability')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Confidence Distribution', fontsize=12, fontweight='bold')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    return results_df

print("✅ Batch test function ready!")

In [ ]:
# =====================================================
# 🔧 EDIT THIS: Enter the filenames you want to test
# =====================================================

TEST_FILES = [
    "329637-8.pck",
    "390116-9.pck",
    "404663-8.pck",
    "406320-9.pck",
    "412857-8.pck",
]  # <-- Add or modify filenames here

# Run batch test
batch_results = test_batch(TEST_FILES, metadata)

---

## 🎯 Option 3: Test Random Samples from Dataset

Quickly test the model on random samples from your dataset.

In [ ]:
def test_random_samples(metadata_df, n_samples=5, data_dir=DATA_DIR):
    """
    Test the model on random samples from the dataset.
    """
    # Filter to files that exist
    available_files = []
    for filename in metadata_df['volumeFilename']:
        if find_pck_file(data_dir, filename):
            available_files.append(filename)
    
    # Random sample
    n_samples = min(n_samples, len(available_files))
    random_files = np.random.choice(available_files, n_samples, replace=False).tolist()
    
    print(f"🎲 Randomly selected {n_samples} scans for testing...")
    return test_batch(random_files, metadata_df, data_dir)

# Run random test
random_results = test_random_samples(metadata, n_samples=5)

---

## 🆕 Option 4: Test Completely New MRI Data

Use this section if you have a new `.pck` file that's NOT in the metadata.

You'll need to provide the ROI coordinates manually.

In [ ]:
def test_new_mri(filepath, roi_info=None):
    """
    Test a completely new MRI file not in the metadata.
    
    Parameters:
    -----------
    filepath : str
        Full path to the .pck file
    roi_info : dict, optional
        ROI coordinates: {'roiZ': z, 'roiY': y, 'roiX': x, 
                          'roiDepth': d, 'roiHeight': h, 'roiWidth': w}
        If None, uses the center of the volume with default size.
    """
    
    print("\n" + "="*60)
    print(f"🆕 TESTING NEW MRI FILE")
    print("="*60)
    print(f"\n📂 File: {filepath}")
    
    if not os.path.exists(filepath):
        print(f"❌ Error: File not found")
        return None
    
    # Load MRI
    print("\n📥 Loading MRI data...")
    mri = load_mri(filepath)
    print(f"   Volume shape: {mri.shape}")
    
    # If no ROI info provided, use center region
    if roi_info is None:
        print("\n⚠️  No ROI info provided - using center of volume")
        d, h, w = mri.shape
        # Default ROI size
        roi_d, roi_h, roi_w = min(32, d), min(64, h), min(64, w)
        
        roi_info = {
            'roiZ': (d - roi_d) // 2,
            'roiY': (h - roi_h) // 2,
            'roiX': (w - roi_w) // 2,
            'roiDepth': roi_d,
            'roiHeight': roi_h,
            'roiWidth': roi_w
        }
        print(f"   Auto ROI: z={roi_info['roiZ']}, y={roi_info['roiY']}, x={roi_info['roiX']}")
        print(f"   Size: {roi_d}x{roi_h}x{roi_w}")
    
    # Extract and process ROI
    roi = extract_roi(mri, roi_info)
    print(f"\n   ROI shape: {roi.shape}")
    
    roi_processed = preprocess_roi(roi)
    tensor = torch.tensor(roi_processed).unsqueeze(0).unsqueeze(0)
    
    # Make prediction
    print("\n🤖 Running prediction...")
    prediction, probability = predict_single(model, tensor, device)
    
    # Display results
    print("\n" + "-"*60)
    print("📊 PREDICTION RESULTS")
    print("-"*60)
    
    if prediction == 1:
        print(f"\n   🔴 ACL TEAR DETECTED")
    else:
        print(f"\n   🟢 NO ACL TEAR DETECTED")
    
    print(f"   Confidence: {probability*100:.1f}%")
    
    # Visualization
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    fig.suptitle(f"New MRI Scan Analysis\n{'🔴 ACL Tear' if prediction == 1 else '🟢 No Tear'} ({probability*100:.1f}% confidence)", 
                 fontsize=14, fontweight='bold')
    
    n_slices = roi_processed.shape[0]
    slice_indices = np.linspace(0, n_slices-1, 8, dtype=int)
    
    for idx, ax in enumerate(axes.flat):
        ax.imshow(roi_processed[slice_indices[idx]], cmap='gray')
        ax.set_title(f"Slice {slice_indices[idx]+1}/{n_slices}")
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'prediction': prediction,
        'probability': probability,
        'label': 'ACL Tear' if prediction == 1 else 'No ACL Tear'
    }

print("✅ New MRI test function ready!")

In [ ]:
# =====================================================
# 🔧 EDIT THIS: Test a completely new MRI file
# =====================================================

# Option A: Provide full path (uncomment to use)
# NEW_MRI_PATH = "path/to/your/new_mri.pck"
# result = test_new_mri(NEW_MRI_PATH)

# Option B: Provide path and ROI coordinates (uncomment to use)
# NEW_MRI_PATH = "path/to/your/new_mri.pck"
# ROI_INFO = {
#     'roiZ': 10,      # Starting Z position
#     'roiY': 50,      # Starting Y position
#     'roiX': 50,      # Starting X position
#     'roiDepth': 20,  # Depth of ROI
#     'roiHeight': 64, # Height of ROI
#     'roiWidth': 64   # Width of ROI
# }
# result = test_new_mri(NEW_MRI_PATH, ROI_INFO)

print("💡 Uncomment the code above and edit the path to test a new MRI file!")

---

## 📝 Quick Reference

| Function | Use Case |
|----------|----------|
| `test_single_scan(filename, metadata)` | Test one scan with detailed visualization |
| `test_batch(filenames, metadata)` | Test multiple scans, get summary report |
| `test_random_samples(metadata, n=5)` | Quick test on random samples |
| `test_new_mri(filepath, roi_info)` | Test completely new MRI file |

### Example Usage:

```python
# Single scan
result = test_single_scan("329637-8.pck", metadata)

# Multiple scans
files = ["file1.pck", "file2.pck", "file3.pck"]
results = test_batch(files, metadata)

# Random samples
results = test_random_samples(metadata, n_samples=10)

# New file
result = test_new_mri("path/to/new/file.pck")
```